In [ ]:
"""
Pydantic AI: full hands-on checklist, AWS Bedrock or Azure OpenAI
================================================================================
Domain: an IT/clinical-systems INCIDENT SEVERITY TRIAGE agent. Every item on the checklist you were given maps to one section below:

  1. Output Schema          -> SeverityResult (Pydantic BaseModel)
  2. System Prompt           -> @agent.instructions
  3. Tool Definition          -> @toolset.tool  (decorator -> agentic action)
  4. Self-Repair Loop          -> ModelRetry raised from a tool + an output_validator
  5. Tool Choice / Retrieval    -> FunctionToolset.filtered(...) — RAG for tools

Model backend is swappable between AWS Bedrock and Azure OpenAI with one
environment variable — this is the "using AWS or Azure" part. Nothing else
in the agent logic changes; that portability is itself a good interview
talking point (Pydantic AI's model abstraction layer).

Install:
    pip install "pydantic-ai-slim[bedrock]"   # for AWS
    pip install "pydantic-ai-slim[openai]"    # for Azure (OpenAI-compatible API)

Run:
    PROVIDER=aws   python 04_pydantic_ai_agent_aws_azure.py
    PROVIDER=azure python 04_pydantic_ai_agent_aws_azure.py
"""

from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Literal

from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry, RunContext, FunctionToolset


# =============================================================================
# 0. MODEL BACKEND — swap AWS Bedrock <-> Azure OpenAI with one env var.
#    This is the "using AWS or Azure Agentic AI" requirement. Everything
#    below (schema, prompt, tools, retry, retrieval) is 100% identical
#    regardless of which branch fires — that portability IS the point.
# =============================================================================
def build_model():
    provider = os.environ.get("PROVIDER", "aws").lower()

    if provider == "aws":
        import boto3
        from botocore.config import Config
        from pydantic_ai.models.bedrock import BedrockConverseModel
        from pydantic_ai.providers.bedrock import BedrockProvider

        bedrock_client = boto3.client(
            "bedrock-runtime",
            region_name=os.environ.get("AWS_REGION", "us-east-1"),
            config=Config(retries={"max_attempts": 5, "mode": "adaptive"}),
        )
        return BedrockConverseModel(
            "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
            provider=BedrockProvider(bedrock_client=bedrock_client),
        )

    elif provider == "azure":
        from pydantic_ai.models.openai import OpenAIChatModel
        from pydantic_ai.providers.azure import AzureProvider

        return OpenAIChatModel(
            "gpt-4o",
            provider=AzureProvider(
                azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],  # e.g. https://your-resource.openai.azure.com/openai/v1/
                api_key=os.environ["AZURE_OPENAI_API_KEY"],
            ),
        )

    raise ValueError(f"Unknown PROVIDER={provider!r}, expected 'aws' or 'azure'")


# =============================================================================
# 1. OUTPUT SCHEMA — force the model into a typed, validated shape instead
#    of free-form text. This is non-negotiable for anything downstream
#    (a dashboard, a paging system) that consumes the agent's output.
# =============================================================================
class SeverityResult(BaseModel):
    level: Literal["P1", "P2", "P3", "P4"] = Field(
        description="P1 = full outage/patient-safety risk, P4 = cosmetic/no impact"
    )
    summary: str = Field(min_length=10, description="One-line incident summary")
    rationale: str = Field(min_length=20, description="Why this severity was assigned")


# Dependencies injected into every tool call — DB clients, config, whatever
# a tool needs. Kept as a plain dataclass so it's fully type-checked.
@dataclass
class IncidentDeps:
    known_systems: dict[str, str]  # system_name -> owning team


# =============================================================================
# 2. SYSTEM PROMPT (INSTRUCTIONS) — persona + hard constraints. Dynamic
#    instructions (via @agent.instructions) can pull in RunContext data,
#    unlike a static string, which is useful when the persona needs
#    runtime facts (e.g. today's on-call rotation).
# =============================================================================
agent = Agent(
    build_model(),
    deps_type=IncidentDeps,
    output_type=SeverityResult,
    retries=2,  # how many times the framework will auto-retry on a ModelRetry
)


@agent.instructions
def system_prompt(ctx: RunContext[IncidentDeps]) -> str:
    return (
        "You are an incident triage agent for Optum IT operations. "
        "Classify incoming incident reports into P1-P4 severity using the "
        "tools available to you before answering. Never guess a system's "
        "owning team — always call lookup_system_owner first. Be concise "
        "and clinical in tone; do not speculate beyond the evidence given. "
        f"You currently have {len(ctx.deps.known_systems)} systems on record."
    )


# =============================================================================
# 3. TOOL DEFINITION — decorators turn plain Python functions into agentic
#    actions. We register a WHOLE toolset (imagine 50+ tools in a real
#    enterprise deployment) so we have something to filter in section 5.
# =============================================================================
incident_tools = FunctionToolset[IncidentDeps]()


@incident_tools.tool
def lookup_system_owner(ctx: RunContext[IncidentDeps], system_name: str) -> str:
    """Return which team owns a given system, e.g. 'claims-api' -> 'Platform Team'."""
    owner = ctx.deps.known_systems.get(system_name)
    if owner is None:
        raise ModelRetry(
            f"'{system_name}' is not a recognized system name. "
            f"Known systems are: {list(ctx.deps.known_systems.keys())}. "
            "Retry with one of these exact names."
        )
    return owner


@incident_tools.tool
def check_recent_incidents(ctx: RunContext[IncidentDeps], system_name: str, hours: int = 24) -> list[str]:
    """Return recent incident titles for a system in the last N hours."""
    if system_name not in ctx.deps.known_systems:
        raise ModelRetry(f"'{system_name}' is not recognized — check lookup_system_owner first.")
    # stand-in for a real incident-management API call
    return [f"[{system_name}] intermittent 500s reported 3h ago"] if hours >= 3 else []


@incident_tools.tool
def page_oncall(ctx: RunContext[IncidentDeps], system_name: str, severity: str) -> str:
    """Page the on-call engineer for a system at a given severity (P1-P4)."""
    # -------------------------------------------------------------------
    # 4. SELF-REPAIR LOOP, first form: validate INSIDE a tool and raise
    #    ModelRetry with a specific, actionable hint. The model sees this
    #    message as the tool's output and gets a chance to correct its
    #    own arguments — no exception surfaces to your application code.
    # -------------------------------------------------------------------
    if severity not in {"P1", "P2", "P3", "P4"}:
        raise ModelRetry(
            f"Invalid severity '{severity}'. Must be exactly one of P1, P2, P3, P4 "
            "(uppercase, no words like 'high' or 'critical')."
        )
    if system_name not in ctx.deps.known_systems:
        raise ModelRetry(f"Unknown system '{system_name}' — call lookup_system_owner first.")
    return f"Paged {ctx.deps.known_systems[system_name]} on-call for {system_name} at {severity}"


# =============================================================================
# 4. SELF-REPAIR LOOP, second form: validate the FINAL structured OUTPUT,
#    not just individual tool calls. If a P1 is claimed without evidence
#    of an actual outage in the rationale, bounce it back for revision.
# =============================================================================
@agent.output_validator
async def validate_severity_result(ctx: RunContext[IncidentDeps], result: SeverityResult) -> SeverityResult:
    if result.level == "P1" and "outage" not in result.rationale.lower() and "down" not in result.rationale.lower():
        raise ModelRetry(
            "You classified this as P1 but the rationale doesn't reference an "
            "outage or system-down evidence. Either lower the severity or cite "
            "concrete evidence for P1 in the rationale."
        )
    return result


# =============================================================================
# 5. TOOL CHOICE / RETRIEVAL — "RAG for tools." Imagine incident_tools had
#    50 tools (paging, ticketing, dashboards, runbooks...). Dumping all 50
#    schemas into every request context window is wasteful and increases
#    the chance the model picks the wrong tool. Instead, score tools
#    against the incoming query and only expose the top-K most relevant
#    ones. Here it's simple keyword overlap; in production you'd embed
#    each tool's description once, embed the incoming query, and do a
#    vector similarity search (exactly like document RAG, but the
#    "documents" are tool descriptions).
# =============================================================================
def relevant_tools_only(query: str, top_k: int = 2):
    """Return a filtered toolset containing only the tools whose name/
    description best match the query — the retrieval step of tool RAG."""
    query_words = set(query.lower().split())

    def score(tool_name: str, description: str) -> int:
        desc_words = set(description.lower().split())
        return len(query_words & desc_words)

    tool_scores = {
        "lookup_system_owner": score(
            "lookup_system_owner", "return which team owns a given system"
        ),
        "check_recent_incidents": score(
            "check_recent_incidents", "return recent incident titles for a system"
        ),
        "page_oncall": score(
            "page_oncall", "page the on-call engineer for a system at a given severity"
        ),
    }
    top_names = {name for name, _ in sorted(tool_scores.items(), key=lambda kv: -kv[1])[:top_k]}

    return incident_tools.filtered(
        lambda ctx, tool_def: tool_def.name in top_names
    )


# =============================================================================
# Wire it together and run.
# =============================================================================
if __name__ == "__main__":
    deps = IncidentDeps(known_systems={
        "claims-api": "Platform Team",
        "member-portal": "Web Team",
        "eligibility-service": "Core Services Team",
    })

    query = "The claims-api is throwing errors and members can't submit claims"

    # Only the top-2 relevant tools get exposed to the model for this run —
    # page_oncall gets filtered OUT here since paging isn't relevant yet.
    scoped_toolset = relevant_tools_only(query, top_k=2)

    result = agent.run_sync(query, deps=deps, toolsets=[scoped_toolset])
    print(result.output)  # -> a validated SeverityResult, guaranteed well-formed

## Production upgrade plan: add MCP + A2A step by step

This version evolves the earlier demo into a production-style architecture in small, layered steps:

1. Step 1 — move the agent behind a service boundary so it can receive incidents through an API or queue.
2. Step 2 — replace local Python tools with MCP-style tool contracts so tools are discoverable and callable in a standardized way.
3. Step 3 — add A2A handoff so the triage agent can delegate to specialist agents like networking, security, or remediation.
4. Step 4 — persist state and emit audit events so the workflow is traceable, retryable, and observable.
5. Step 5 — add approval gates and safety rules for high-impact actions such as paging or remediation.

Each step is added incrementally so the system can grow from a single-agent demo into a governed multi-agent workflow.